# Phase 4b (Colab T4): why is llama.cpp slow under load?

**Question:** in Phase 4, llama.cpp (GGUF Q4_K_M) was fastest for one request but its throughput barely grew with load (about 48 tokens/s with 1 request, only about 95 with 32), while vLLM scaled to 400+. The server log showed requests were *not* queued, so the cause is how the server batches work. This notebook tests the likely fixes.

**What it does:** starts llama.cpp with different settings (number of parallel slots, flash attention, batch sizes), runs a short load test on each, and prints one comparison table. Everything is measured on **this** GPU (a T4), so compare rows with each other, **not** with the L4 numbers from Phase 4.

| Setting name | What changes |
|---|---|
| `baseline-np64` | exactly what Phase 4 used: 64 parallel slots |
| `np16`, `np8` | fewer slots (less work per step) |
| `np1` | a single slot (the "no batching at all" reference) |
| `np16-fa-off` | flash attention switched off (default is automatic) |
| `np16-ub1024` | larger prompt-processing batch |
| *control (optional)* | a plain-attention model (Qwen3-4B, not our fine-tune) with the same settings: if it scales much better, the hybrid Qwen3.5 layers are the cause |

**How to read the result:** if *no* setting makes throughput grow much with load, the limit is inside llama.cpp's handling of the model, not a flag.

**Before you run:** runtime = T4 GPU; Colab secret `HF_TOKEN` (read access, Notebook access on); `banking-finetuning-code.zip` in the root of your Drive. About 45 minutes in total. Results are saved to Drive folder `Banking_FInetuning_phase4/tuning/`.

In [ ]:
from google.colab import drive, files, userdata
drive.mount('/content/drive')
import os, glob, shutil, subprocess, sys, json, time, signal
import requests

RES = '/content/drive/MyDrive/Banking_FInetuning_phase4/tuning'
os.makedirs(RES, exist_ok=True)
def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout

GPU_NAME, GPU_CC, GPU_MIB = [x.strip() for x in sh('nvidia-smi --query-gpu=name,compute_cap,memory.total --format=csv,noheader,nounits').strip().splitlines()[0].split(',')]
print('GPU:', GPU_NAME, '| compute capability', GPU_CC, '|', GPU_MIB, 'MiB')
json.dump({'gpu': GPU_NAME, 'compute_cap': GPU_CC, 'memory_mib': int(GPU_MIB)}, open(f'{RES}/gpu.json', 'w'))

cands = glob.glob('/content/drive/MyDrive/banking-finetuning-code.zip') + glob.glob('/content/drive/MyDrive/*/banking-finetuning-code.zip')
zpath = cands[0] if cands else list(files.upload())[0]
PROJ = '/content/Banking_FInetuning'
shutil.rmtree(PROJ, ignore_errors=True)
shutil.unpack_archive(zpath, PROJ)
os.chdir(PROJ)
assert os.path.exists('loadtest/bench_raw.py'), 'Old code zip: upload the newest banking-finetuning-code.zip'
print('code from', zpath, '| results go to', RES)

In [ ]:
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception as e:
    raise RuntimeError('Add a Colab secret named HF_TOKEN (key icon, left bar) and enable Notebook access, then re-run this cell') from e
os.environ['HF_TOKEN'] = HF_TOKEN

## 1. Install, data, download
No vLLM this time, so the install is small. Only the Q4_K_M file (2.7 GB) and the tokenizer files are downloaded.

In [ ]:
!pip install -q -U transformers huggingface_hub httpx pandas scikit-learn pyyaml datasets
if not os.path.exists('data/test.jsonl'):
    r = subprocess.run([sys.executable, '-m', 'src.data.prepare'], capture_output=True, text=True)
    print(r.stdout[-500:], r.stderr[-800:] if r.returncode else '')
print(sh('wc -l data/test.jsonl'))

In [ ]:
from huggingface_hub import snapshot_download
REPO = 'Abhishek-255/banking-triage-qwen3_5_4b-quantized'
MODELS = '/content/models'
os.makedirs(MODELS, exist_ok=True)
snapshot_download(REPO, local_dir=MODELS, token=HF_TOKEN, allow_patterns=[
    'gguf/model-Q4_K_M.gguf', 'merged-fp16/tokenizer*', 'merged-fp16/vocab.json', 'merged-fp16/merges.txt',
    'merged-fp16/chat_template*', 'merged-fp16/special_tokens_map.json', 'merged-fp16/added_tokens.json', 'merged-fp16/config.json'])
GGUF = f'{MODELS}/gguf/model-Q4_K_M.gguf'
TOK = f'{MODELS}/merged-fp16'
assert os.path.exists(GGUF), 'GGUF not found in the repo'
print(sh(f'ls -la {MODELS}/gguf {TOK}'))

## 2. Build llama.cpp for this GPU
About 10 minutes with no output while it compiles; the architecture (75 for a T4) is detected automatically.

In [ ]:
LLAMA, LLAMA_BIN = '/content/llama.cpp', '/content/llama.cpp/build/bin'
if not os.path.exists(LLAMA):
    subprocess.run(['git', 'clone', '-q', '--depth', '1', 'https://github.com/ggml-org/llama.cpp', LLAMA], check=True)
cands = (glob.glob('/usr/local/cuda*/lib64/stubs/libcuda.so') + glob.glob('/usr/lib/x86_64-linux-gnu/libcuda.so*')
         + glob.glob('/usr/lib64-nvidia/libcuda.so*') + glob.glob('/usr/local/nvidia/lib64/libcuda.so*'))
print('libcuda candidates:', cands)
assert cands, 'No libcuda found: copy this message to Claude'
STUB = '/content/cudastub'
os.makedirs(STUB, exist_ok=True)
if os.path.lexists(f'{STUB}/libcuda.so'):
    os.remove(f'{STUB}/libcuda.so')
os.symlink(cands[0], f'{STUB}/libcuda.so')
shutil.rmtree(f'{LLAMA}/build', ignore_errors=True)
cfg = (f'cmake -S {LLAMA} -B {LLAMA}/build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES={GPU_CC.replace(".", "")} -DCMAKE_LIBRARY_PATH={STUB} '
       f'-DLLAMA_CURL=OFF -DCMAKE_BUILD_TYPE=Release > /content/cmake.log 2>&1')
rc = subprocess.run(cfg, shell=True).returncode
print('cmake configure exit code:', rc)
if rc:
    print(sh("grep -n -i -B2 -A6 -E 'CMake Error|could not|not found|missing' /content/cmake.log | head -60"))
    raise RuntimeError('llama.cpp configure failed: copy the errors above to Claude')
rc = subprocess.run(f'cmake --build {LLAMA}/build --target llama-server -j{os.cpu_count()} > /content/build.log 2>&1', shell=True).returncode
print('build exit code:', rc)
if rc:
    print(sh("grep -n -i -B3 -A3 -E 'error' /content/build.log | head -50"))
    raise RuntimeError('llama.cpp build failed: copy the errors above to Claude')
assert os.path.exists(f'{LLAMA_BIN}/llama-server')
print('built llama-server')

## 3. Helpers
`run_cfg(name, extra_flags)` starts the server with the given flags, runs the load test at 1, 8 and 32 simultaneous requests, saves the server log, and stops the server.

In [ ]:
PORT = 8000
CSV = f'{RES}/sweep_tuning.csv'

def gpu_used_mib():
    return int(sh('nvidia-smi --query-gpu=memory.used --format=csv,noheader,nounits').strip().splitlines()[0])

def start_llama(gguf, extra, log_path):
    cmd = [f'{LLAMA_BIN}/llama-server', '-m', gguf, '--alias', 'banking', '--host', '127.0.0.1', '--port', str(PORT), '-ngl', '99'] + extra
    return subprocess.Popen(cmd, stdout=open(log_path, 'w'), stderr=subprocess.STDOUT, preexec_fn=os.setsid)

def wait_ready(p, log_path, timeout=600):
    t0 = time.time()
    while time.time() - t0 < timeout:
        if p.poll() is not None:
            raise RuntimeError('server exited. Last log lines:' + chr(10) + sh(f'tail -40 {log_path}'))
        try:
            if requests.get(f'http://127.0.0.1:{PORT}/health', timeout=2).status_code == 200:
                return
        except Exception:
            pass
        time.sleep(3)
    raise RuntimeError('server not ready in time. Last log lines:' + chr(10) + sh(f'tail -40 {log_path}'))

def stop(p):
    try:
        os.killpg(os.getpgid(p.pid), signal.SIGTERM)
        p.wait(timeout=40)
    except Exception:
        try:
            os.killpg(os.getpgid(p.pid), signal.SIGKILL)
        except Exception:
            pass
    sh('pkill -f llama-server')
    for _ in range(40):
        if gpu_used_mib() < 1500:
            return
        time.sleep(2)
    print('WARNING: GPU memory not fully released:', gpu_used_mib(), 'MiB')

def run_cfg(name, extra, gguf=None, tok=None, levels='1,8,32', max_tokens=64):
    gguf, tok = gguf or GGUF, tok or TOK
    log = f'/content/{name}.log'
    print('=== ', name, extra)
    p = start_llama(gguf, extra, log)
    try:
        t0 = time.time()
        wait_ready(p, log)
        print(f'server ready in {time.time() - t0:.0f}s')
        r = subprocess.run([sys.executable, 'loadtest/bench_raw.py', '--base-url', f'http://127.0.0.1:{PORT}', '--label', name,
                            '--tokenizer', tok, '--levels', levels, '--min-requests', '150', '--max-tokens', str(max_tokens),
                            '--out-csv', CSV], capture_output=True, text=True)
        print(r.stdout[-1800:])
        if r.returncode:
            print('LOAD TEST FAILED:', r.stderr[-1500:])
    finally:
        stop(p)
        shutil.copy(log, f'{RES}/{name}.log')

print('GPU:', GPU_NAME)

## 4. The tuning runs
One cell per setting (about 5 minutes each). Run the first one before anything else: it is the reference all others are compared against.

In [ ]:
run_cfg('baseline-np64', ['-np', '64', '-c', '65536'])

In [ ]:
run_cfg('np16', ['-np', '16', '-c', '16384'])

In [ ]:
run_cfg('np8', ['-np', '8', '-c', '8192'])

In [ ]:
run_cfg('np1', ['-np', '1', '-c', '2048'])

In [ ]:
run_cfg('np16-fa-off', ['-np', '16', '-c', '16384', '-fa', 'off'])

In [ ]:
run_cfg('np16-ub1024', ['-np', '16', '-c', '16384', '-b', '2048', '-ub', '1024'])

## 5. Control (optional): a plain-attention model
Same server settings, but a standard Qwen3-4B (all layers are ordinary attention; not our fine-tune, so ignore its answers; only speed matters). If this model scales much better with load than our Qwen3.5 model under identical settings, the hybrid layers are the cause. Skip this section if the download fails; the tuning results above stand on their own.

In [ ]:
CTRL_OK = False
try:
    from huggingface_hub import hf_hub_download
    CTRL_GGUF = hf_hub_download('unsloth/Qwen3-4B-GGUF', 'Qwen3-4B-Q4_K_M.gguf', local_dir='/content/models/ctrl')
    CTRL_TOK = snapshot_download('Qwen/Qwen3-4B', local_dir='/content/models/ctrl_tok',
                                 allow_patterns=['tokenizer*', 'vocab.json', 'merges.txt', 'config.json', 'generation_config.json'])
    CTRL_OK = True
    print('control model ready:', CTRL_GGUF)
except Exception as e:
    print('control model not available, skipping:', repr(e)[:300])

In [ ]:
if CTRL_OK:
    run_cfg('control-qwen3-np64', ['-np', '64', '-c', '65536'], gguf=CTRL_GGUF, tok=CTRL_TOK, max_tokens=24)
    run_cfg('control-qwen3-np16', ['-np', '16', '-c', '16384'], gguf=CTRL_GGUF, tok=CTRL_TOK, max_tokens=24)

## 6. Results
Output tokens per second at each load level (higher is better), plus tail latency. The key number is the **scaling** column: tokens/s at 32 simultaneous requests divided by tokens/s at 1 request. vLLM on the L4 in Phase 4 scaled about 15x (27 to 412).

In [ ]:
import pandas as pd
pd.set_option('display.width', 200)
df = pd.read_csv(CSV)
tok = df.pivot(index='variant', columns='concurrency', values='out_tok_per_s')
p95 = df.pivot(index='variant', columns='concurrency', values='p95_s')
out = tok.add_prefix('tok/s @').join(p95.add_prefix('p95 s @'))
out['scaling 32 vs 1'] = (tok[32] / tok[1]).round(2)
print(out.round(2).to_string())
print()
print('errors per variant:', df.groupby('variant')['errors'].sum().to_dict())
out.round(3).to_csv(f'{RES}/tuning_summary.csv')
print('saved', f'{RES}/tuning_summary.csv')

In [ ]:
zp = shutil.make_archive('/content/phase4b_tuning_results', 'zip', RES)
print(zp)
files.download(zp)